# Multi-Turn Sessions

Continue conversations across multiple requests. Jockey maintains context within a session, enabling iterative exploration and drill-down queries.

In [ ]:
import json
import os
import time

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

# Replace with your knowledge store ID
STORE_ID = "your_knowledge_store_id"

## When You Need This

- Drilling down into findings ("tell me more about that second theme")
- Iterative exploration of a video collection
- Building chat interfaces

## Helper: Parse Response

In [ ]:
def parse_response(result: dict) -> str | dict:
    """Extract text content from a Jockey API response.

    Args:
        result: The parsed JSON response from the API.

    Returns:
        The text content string, or an empty string if not found.
    """
    for output in result["output"]:
        if output["type"] == "message":
            for content in output["content"]:
                return content["text"]
    return ""

## Starting a Session

Omit `session_id` on your first request. Jockey creates a new session and returns its ID in the response.

In [ ]:
# First message -- new session
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "What are the main themes in these videos?",
            }
        ],
        "knowledge_store_id": STORE_ID,
    },
)

result = response.json()
session_id = result["session_id"]
print(f"Session started: {session_id}")
print(f"Response: {parse_response(result)[:200]}...")

## Continuing the Conversation

Pass `session_id` to follow up. Jockey remembers the full conversation context server-side -- you do not need to resend previous messages.

In [ ]:
# Second message -- same session
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "session_id": session_id,
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "Which videos best represent the first theme?",
            }
        ],
        "knowledge_store_id": STORE_ID,
    },
)

result = response.json()
print(f"Turn 2: {parse_response(result)[:200]}...")

In [ ]:
# Third message -- drilling deeper
response = requests.post(
    f"{BASE_URL}/responses",
    headers=HEADERS,
    json={
        "model": "jockey1.0",
        "session_id": session_id,
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": "Give me timestamps for the key moments in those videos",
            }
        ],
        "knowledge_store_id": STORE_ID,
    },
)

result = response.json()
print(f"Turn 3: {parse_response(result)[:200]}...")

## Session Pattern for Chat UIs

A reusable `chat` function that automatically manages the session ID across turns. This pattern is ideal for building chat-style interfaces.

In [ ]:
class ChatSession:
    """Manages a multi-turn Jockey conversation session."""

    def __init__(self, store_id: str) -> None:
        self.store_id = store_id
        self.session_id: str | None = None

    def chat(self, message: str) -> str:
        """Send a message and return the assistant response.

        Args:
            message: The user message to send.

        Returns:
            The assistant's text response.
        """
        payload: dict = {
            "model": "jockey1.0",
            "input": [
                {"type": "message", "role": "user", "content": message}
            ],
            "knowledge_store_id": self.store_id,
        }

        if self.session_id:
            payload["session_id"] = self.session_id

        response = requests.post(
            f"{BASE_URL}/responses",
            headers=HEADERS,
            json=payload,
        )

        result = response.json()
        self.session_id = result["session_id"]

        return parse_response(result)

In [ ]:
# Example usage of the ChatSession class
session = ChatSession(store_id=STORE_ID)

print("Turn 1:", session.chat("What are the main themes in these videos?")[:200])
print(f"\n(session_id: {session.session_id})")

print("\nTurn 2:", session.chat("Tell me more about the first theme")[:200])
print("\nTurn 3:", session.chat("Which video is the best example?")[:200])

## Common Pitfalls

- **Store the `session_id`** -- if you lose it, you cannot continue the conversation
- **Same knowledge store** -- keep the same `knowledge_store_id` across turns for consistent results
- **Session state is server-side** -- you do not need to resend conversation history

## Next Steps

- [Streaming](./streaming.ipynb) -- Receive responses in real-time via SSE
- [Structured Output](./structured_output.ipynb) -- Force Jockey to return typed JSON
- [Error Handling](./error_handling.ipynb) -- Retry strategies, polling helpers, and common error patterns
- [API Reference: POST /responses](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/responses/create-response)